<div style="display:flex; align-items:flex-start; gap:2rem; padding:1rem 0;">

  <div style="flex-shrink:0;">
    <img src="https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"
         style="height:36px; display:block; margin-bottom:0.75rem;"/>
    <img src="https://i.imgur.com/ITL8dZE.jpeg"
         style="width:260px; border-radius:4px; box-shadow:0 2px 8px rgba(0,0,0,0.15); display:block;"/>
  </div>

  <div style="flex:1; font-family:sans-serif;">
    <h1 style="font-size:1.6rem; font-weight:600; margin:0 0 0.25rem 0;">
      AI, ML and GenAI in the Lakehouse
    </h1>
    <p style="margin:0 0 0.75rem 0; color:#555; font-size:0.95rem;">
      <strong>Chapter 07-01 &mdash; Machine Learning at Scale with the NYC Taxi Dataset</strong><br/>
      Author: Bennie Haelen &nbsp;&bull;&nbsp; Date: 10-28-2024
    </p>
    <p style="margin:0 0 0.75rem 0; color:#333; font-size:0.9rem;">
      Demonstrates machine learning at scale using Apache Spark MLlib and Delta Lake
      on the full NYC Yellow Taxi 2023 dataset, including distributed GBT training,
      Hyperopt tuning with SparkTrials, and MLflow tracking with Unity Catalog registration.
    </p>
    <table style="border-collapse:collapse; font-size:0.85rem; color:#333; width:100%;">
      <tr style="background:#f5f5f5;">
        <td style="padding:4px 10px; font-weight:600; white-space:nowrap;">Section</td>
        <td style="padding:4px 10px; font-weight:600;">Steps</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>0 &mdash; Setup</strong></td>
        <td style="padding:4px 10px;">Constants &bull; imports &bull; MLflow registry URI</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>1 &mdash; Data ingestion</strong></td>
        <td style="padding:4px 10px;">Load 12 monthly Parquet files &bull; estimate row count &bull; type conversions</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>2 &mdash; Preprocessing</strong></td>
        <td style="padding:4px 10px;">Trip duration &bull; filter outliers &bull; drop nulls</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>3 &mdash; Feature engineering</strong></td>
        <td style="padding:4px 10px;">Temporal features &bull; numerical imputation &bull; categorical fill</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>4 &mdash; Vectorization</strong></td>
        <td style="padding:4px 10px;">StringIndexer &bull; OneHotEncoder &bull; VectorAssembler &bull; finalise dataset</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>5 &mdash; Delta Lake storage</strong></td>
        <td style="padding:4px 10px;">Write to Delta &bull; OPTIMIZE</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>6 &mdash; Model training</strong></td>
        <td style="padding:4px 10px;">Train/test split &bull; GBT baseline &bull; evaluate RMSE</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>7 &mdash; MLlib cross-validation</strong></td>
        <td style="padding:4px 10px;">ParamGridBuilder &bull; CrossValidator &bull; optimised RMSE</td>
      </tr>
      <tr>
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>8 &mdash; Hyperopt tuning</strong></td>
        <td style="padding:4px 10px;">Objective function &bull; search space &bull; SparkTrials &bull; MLflow tracking</td>
      </tr>
      <tr style="background:#f9f9f9;">
        <td style="padding:4px 10px; white-space:nowrap; vertical-align:top;"><strong>9 &mdash; Register in Unity Catalog</strong></td>
        <td style="padding:4px 10px;">Log best model &bull; register &bull; set champion alias &bull; verify</td>
      </tr>
    </table>
  </div>

</div>

# Section 0: Setup and Prerequisites

## 0-1: Constants
Run the shared constants notebook if present, otherwise set values inline below.

In [0]:
%run "../common/Constants"

In [0]:
# Inline fallback -- update these paths if the %run cell above is commented out.
# NYC_TAXI_DATASET_PATH   = "/mnt/nyc_taxi/raw"
# DELTA_NYC_DATASET_PATH  = "/mnt/nyc_taxi/delta"
# CATALOG_NAME            = "book_ai_ml_lakehouse"
# ML_SCHEMA              = "ml_models"

## 0-2: Imports

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.functions import (
    col, isnan, when, count,
    date_format, hour, dayofweek, unix_timestamp
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import mlflow
import mlflow.spark

## 0-3: Configure MLflow for Unity Catalog
Point MLflow at Unity Catalog so all model registrations go to the three-level namespace
rather than the legacy Workspace Model Registry.

In [0]:
# All mlflow.register_model() calls in this notebook will target Unity Catalog.
mlflow.set_registry_uri("databricks-uc")

# Section 1: Data Ingestion and Initial Exploration

## 1-1: Load the dataset

In [0]:
# Load all 12 monthly Parquet files for the 2023 NYC Yellow Taxi dataset.
# Each file follows the naming convention yellow_tripdata_2023_MM.parquet.
files = [
    f"dbfs:{NYC_TAXI_DATASET_PATH}/yellow_tripdata_2023_{i:02}.parquet"
    for i in range(1, 13)
]

dfs = []
for file in files:
    df = spark.read.parquet(file)
    # Normalise passenger_count to double across all monthly files
    # before the union -- some months encode it as long, others as double.
    df = df.withColumn("passenger_count", df["passenger_count"].cast("double"))
    dfs.append(df)

# Union all monthly DataFrames into one.
taxi_df = dfs[0]
for df in dfs[1:]:
    taxi_df = taxi_df.union(df)

taxi_df.printSchema()
display(taxi_df)

## 1-2: Estimate row count by sampling

In [0]:
# Counting all rows on a full-year dataset can take several minutes.
# A 10% sample gives a fast estimate with ~1% error at this scale.
sample_fraction = 0.1
sample_count = taxi_df.sample(fraction=sample_fraction, seed=42).count()
estimated_total = int(sample_count / sample_fraction)
print(f"Sampled rows:      {sample_count:,}")
print(f"Estimated total:   {estimated_total:,}")

## 1-3: Type conversions
The Parquet files encode several columns as `long` or `double` where a narrower
integer or float type is more appropriate and uses less memory across the cluster.

In [0]:
taxi_df = (
    taxi_df
    .withColumn("VendorID",               col("VendorID").cast("integer"))
    .withColumn("tpep_pickup_datetime",   col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime",  col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("passenger_count",        col("passenger_count").cast("integer"))
    .withColumn("trip_distance",          col("trip_distance").cast("float"))
    .withColumn("RatecodeID",             col("RatecodeID").cast("integer"))
    .withColumn("PULocationID",           col("PULocationID").cast("integer"))
    .withColumn("DOLocationID",           col("DOLocationID").cast("integer"))
    .withColumn("payment_type",           col("payment_type").cast("integer"))
    .withColumn("fare_amount",            col("fare_amount").cast("float"))
    .withColumn("extra",                  col("extra").cast("float"))
    .withColumn("mta_tax",                col("mta_tax").cast("float"))
    .withColumn("tip_amount",             col("tip_amount").cast("float"))
    .withColumn("tolls_amount",           col("tolls_amount").cast("float"))
    .withColumn("improvement_surcharge",  col("improvement_surcharge").cast("float"))
    .withColumn("total_amount",           col("total_amount").cast("float"))
    .withColumn("congestion_surcharge",   col("congestion_surcharge").cast("float"))
    .withColumn("airport_fee",            col("airport_fee").cast("float"))
)

taxi_df.printSchema()
display(taxi_df)

# Section 2: Data Preprocessing at Scale

## 2-1: Add a trip_duration column

In [0]:
# Compute trip duration in minutes from the pickup and drop-off timestamps.
# unix_timestamp() converts each timestamp to seconds since epoch;
# subtracting and dividing by 60 gives duration in minutes.
taxi_df = taxi_df.withColumn(
    "trip_duration",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

## 2-2: Filter out trips with unrealistic durations or distances

In [0]:
# Remove trips with zero, negative, or excessively long durations (> 4 hours).
taxi_df = taxi_df.filter((col("trip_duration") > 0) & (col("trip_duration") <= 240))

# Remove trips with zero, negative, or implausibly long distances (> 100 miles).
taxi_df = taxi_df.filter((col("trip_distance") > 0) & (col("trip_distance") <= 100))

print(f"Rows after filtering: {taxi_df.count():,}")

## 2-3: Drop rows with missing critical values

In [0]:
# Drop any row where a column that is essential for training or evaluation is null.
taxi_df = taxi_df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "trip_duration",
])

print(f"Rows after null drop: {taxi_df.count():,}")

# Section 3: Feature Engineering

## 3-1: Extract temporal features

In [0]:
from pyspark.sql.functions import col, hour, dayofweek, date_format
from pyspark.sql.types import IntegerType

# Hour of day (0-23): captures peak vs off-peak demand.
taxi_df = taxi_df.withColumn("pickup_hour",  hour(col("tpep_pickup_datetime")))

# Day of week (1=Sunday ... 7=Saturday): captures weekday vs weekend patterns.
taxi_df = taxi_df.withColumn("pickup_day",   dayofweek(col("tpep_pickup_datetime")))

# Month (1-12): captures seasonal demand shifts across the year.
taxi_df = taxi_df.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "M").cast(IntegerType())
)

# Year: included for completeness; all rows are 2023 in this dataset.
taxi_df = taxi_df.withColumn(
    "pickup_year",
    date_format(col("tpep_pickup_datetime"), "yyyy").cast(IntegerType())
)

## 3-2: Impute missing values in numerical features

In [0]:
# Replace any remaining nulls in numerical feature columns with the column mean.
# Imputer operates in a distributed manner on the Spark DataFrame.
numerical_cols = ["trip_distance", "passenger_count", "pickup_hour"]

imputer = Imputer(
    inputCols=numerical_cols,
    outputCols=numerical_cols
).setStrategy("mean")

taxi_df = imputer.fit(taxi_df).transform(taxi_df)

## 3-3: Fill missing values in categorical features

In [0]:
# Use -1 as a sentinel for missing categorical values.
# StringIndexer's handleInvalid='keep' will assign these a dedicated index slot
# rather than failing or dropping the row.
categorical_columns = ["PULocationID", "DOLocationID", "pickup_day"]

taxi_df = taxi_df.fillna({col_name: -1 for col_name in categorical_columns})

# Section 4: Feature Transformation and Vectorization

## 4-1: Set up StringIndexers and OneHotEncoders

In [0]:
# StringIndexer converts each categorical column to a numeric index.
# OneHotEncoder converts that index to a sparse binary vector.
# handleInvalid='keep' creates an extra slot for unseen or null values at inference time.
indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_indexed",
        handleInvalid="keep"
    )
    for col_name in categorical_columns
]

encoders = [
    OneHotEncoder(
        inputCol=col_name + "_indexed",
        outputCol=col_name + "_encoded",
        handleInvalid="keep"
    )
    for col_name in categorical_columns
]

print(f"Indexers: {len(indexers)}  |  Encoders: {len(encoders)}")

## 4-2: Set up the VectorAssembler

In [0]:
# The assembler combines all encoded categorical columns and numerical columns
# into a single dense/sparse 'features' vector required by Spark MLlib.
assembler_input_cols = (
    [col_name + "_encoded" for col_name in categorical_columns]
    + numerical_cols
)

assembler = VectorAssembler(
    inputCols=assembler_input_cols,
    outputCol="features",
    handleInvalid="skip"   # skip rows where any input column is still null
)

## 4-3: Execute the indexers

In [0]:
df_indexed = taxi_df
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

df_indexed.printSchema()

## 4-4: Execute the encoders

In [0]:
df_encoded = df_indexed
for encoder in encoders:
    df_encoded = encoder.fit(df_encoded).transform(df_encoded)

df_encoded.printSchema()

## 4-5: Assemble the features column

In [0]:
df_assembled = assembler.transform(df_encoded)
df_assembled.printSchema()

## 4-6: Finalise the dataset
Select only the `features` vector and the target variable, renaming `trip_duration`
to `label` so it matches the default expected by Spark MLlib estimators.

In [0]:
df_final = (
    df_assembled
    .select("features", "trip_duration")
    .withColumnRenamed("trip_duration", "label")
)

print(df_final.columns)
display(df_final.limit(5))

# Section 5: Efficient Storage with Delta Lake

## 5-1: Write to Delta Lake

In [0]:
delta_path = f"dbfs:{DELTA_NYC_DATASET_PATH}"
print(f"Writing to: {delta_path}")

# Remove any previous version so the overwrite starts clean.
dbutils.fs.rm(delta_path, recurse=True)

# Write in Delta format. mode('overwrite') replaces any existing data.
df_final.write.format("delta").mode("overwrite").save(delta_path)
print("Write complete.")

## 5-2: Optimize the Delta table
`OPTIMIZE` compacts small files written during the distributed write into larger
files, reducing the number of read operations during model training.

In [0]:
# OPTIMIZE using the file path -- single-quoted path in SQL is correct here.
spark.sql(f"OPTIMIZE delta.`{DELTA_NYC_DATASET_PATH}`")
print("OPTIMIZE complete.")

# Section 6: Model Training at Scale

## 6-1: Read back from Delta and split the data

In [0]:
# Read the optimised Delta table back into a Spark DataFrame.
df_model = spark.read.format("delta").load(delta_path)

# 80/20 train/test split with a fixed seed for reproducibility.
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_data.count():,}")
print(f"Test rows:     {test_data.count():,}")

## 6-2: Train a Gradient Boosted Tree regressor
GBTRegressor is a distributed ensemble method in Spark MLlib that builds an
ensemble of decision trees iteratively, with each tree correcting the errors of
the previous one. It handles non-linear relationships and is well-suited to
tabular regression tasks at the scale of the NYC Taxi dataset.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS book_ai_ml_lakehouse.ml_models; CREATE VOLUME IF NOT EXISTS book_ai_ml_lakehouse.ml_models.tmp;

In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(featuresCol="features", labelCol="label", maxIter=5)

train_data.cache()
train_data.count()

with mlflow.start_run(run_name="gbt_baseline"):
    gbt_model = gbt.fit(train_data)
    mlflow.log_param("maxIter",  gbt.getMaxIter())
    mlflow.log_param("maxDepth", gbt.getMaxDepth())
    mlflow.spark.log_model(
        gbt_model,
        artifact_path="model",
        dfs_tmpdir="/Volumes/book_ai_ml_lakehouse/ml_models/tmp"
    )
    print("Training complete.")

## 6-3: Evaluate the baseline model

In [0]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

predictions = gbt_model.transform(test_data)
rmse = evaluator.evaluate(predictions)
print(f"Baseline GBT RMSE: {rmse:.2f} minutes")

# Section 7: Hyperparameter Tuning with MLlib CrossValidator
Spark MLlib's `CrossValidator` and `ParamGridBuilder` provide an exhaustive grid
search with k-fold cross-validation, running entirely within the Spark execution
engine across the cluster.

In [0]:
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth,  [5, 7, 9])
    .addGrid(gbt.maxIter,   [50, 100])
    .addGrid(gbt.stepSize,  [0.05, 0.1, 0.2])
    .build()
)

cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

with mlflow.start_run(run_name="gbt_crossval"):
    cv_model = cv.fit(train_data)

    cv_predictions = cv_model.transform(test_data)
    cv_rmse = evaluator.evaluate(cv_predictions)

    best_gbt = cv_model.bestModel
    mlflow.log_param("best_maxDepth", best_gbt.getMaxDepth())
    mlflow.log_param("best_maxIter",  best_gbt.getNumTrees())
    mlflow.log_metric("rmse", cv_rmse)

    input_example = train_data.limit(5).toPandas()
    mlflow.spark.log_model(
        best_gbt,
        artifact_path="model",
        dfs_tmpdir="/Volumes/book_ai_ml_lakehouse/ml_models/tmp",
        input_example=input_example
    )

print(f"CrossValidator best RMSE: {cv_rmse:.2f} minutes")
print(f"Best maxDepth: {best_gbt.getMaxDepth()}")
print(f"Best maxIter:  {best_gbt.getNumTrees()}")
print(f"Best maxIter:  {best_gbt.getNumTrees()}")

# Section 8: Distributed Hyperparameter Tuning with Hyperopt and SparkTrials
`SparkTrials` distributes each Hyperopt trial as a separate Spark task, running
multiple trials in parallel across the cluster. Combined with MLflow nested runs,
each trial is tracked individually under a parent experiment run.

## 8-1: Set recursion limit for Hyperopt

In [0]:
import sys
# Hyperopt uses recursive search algorithms that can exceed Python's default limit.
sys.setrecursionlimit(10000)

## 8-2: Define the objective function

In [0]:
from hyperopt import STATUS_OK

def objective_function(params):
    """
    Train a GBTRegressor with the given hyperparameters, evaluate on the test set,
    and return the RMSE as the loss for Hyperopt to minimise.
    Each call is logged as a nested MLflow child run.
    """
    max_depth = int(params["maxDepth"])
    max_iter  = int(params["maxIter"])

    with mlflow.start_run(nested=True):
        mlflow.log_params({"maxDepth": max_depth, "maxIter": max_iter})

        gbt_trial = GBTRegressor(
            featuresCol="features",
            labelCol="label",
            maxDepth=max_depth,
            maxIter=max_iter
        )

        model = gbt_trial.fit(train_data)
        preds = model.transform(test_data)
        rmse  = evaluator.evaluate(preds)

        mlflow.log_metric("rmse", rmse)

    return {"loss": rmse, "status": STATUS_OK}

## 8-3: Define the search space

In [0]:
from hyperopt import hp

# quniform produces discrete values: maxDepth sampled from {5,6,...,15},
# maxIter sampled from {50,60,...,150}.
search_space = {
    "maxDepth": hp.quniform("maxDepth", 5, 15, 1),
    "maxIter":  hp.quniform("maxIter",  50, 150, 10),
}

## 8-4: Run Hyperopt with SparkTrials

In [0]:
from hyperopt import fmin, tpe, SparkTrials

# parallelism=4 runs 4 trials concurrently on the cluster.
# Increase this value if your cluster has more workers available.
spark_trials = SparkTrials(parallelism=4)

with mlflow.start_run(run_name="hyperopt_gbt_tuning"):
    best_hyperparams = fmin(
        fn=objective_function,
        space=search_space,
        algo=tpe.suggest,
        max_evals=20,
        trials=spark_trials
    )

print(f"Best hyperparameters: {best_hyperparams}")

# Section 9: Log and Register the Best Model in Unity Catalog
Logging through `mlflow.spark.log_model()` and registering in Unity Catalog creates
a governed, versioned artifact with full lineage back to the training run. The model
alias 'champion' replaces the deprecated stage-transition API.

## 9-1: Train the best model and log to Unity Catalog

In [0]:
# Build the Unity Catalog three-level model name.
model_name       = "nyc_taxi_trip_duration_model"
unity_model_name = f"{CATALOG_NAME}.{ML_SCHEMA}.{model_name}"
print(f"Registering as: {unity_model_name}")

In [0]:
# Train a final model using the best hyperparameters found by Hyperopt.
best_max_depth = int(best_hyperparams["maxDepth"])
best_max_iter  = int(best_hyperparams["maxIter"])

gbt_best = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=best_max_depth,
    maxIter=best_max_iter
)

with mlflow.start_run(run_name="gbt_best_model") as run:

    mlflow.log_param("maxDepth", best_max_depth)
    mlflow.log_param("maxIter",  best_max_iter)

    # Train and evaluate.
    gbt_best_model = gbt_best.fit(train_data)
    best_preds     = gbt_best_model.transform(test_data)
    best_rmse      = evaluator.evaluate(best_preds)

    mlflow.log_metric("rmse", best_rmse)
    print(f"Best model RMSE: {best_rmse:.2f} minutes")

    # Log and register in Unity Catalog in one call.
    mlflow.spark.log_model(
        gbt_best_model,
        artifact_path="model",
        registered_model_name=unity_model_name,
    )

print(f"Model registered: {unity_model_name}")

## 9-2: Assign the 'champion' alias
Model stage transitions (`transition_model_version_stage`) are deprecated in
MLflow 2.x and Unity Catalog. The recommended replacement is model aliases,
which serve the same purpose without the fixed stage vocabulary.

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Retrieve the latest version just registered.
latest_versions = client.get_registered_model(unity_model_name).latest_versions
latest_version  = max(int(v.version) for v in latest_versions)

# Set the 'champion' alias so downstream consumers can load
# models://<catalog>.<schema>.<model>@champion without hardcoding a version number.
client.set_registered_model_alias(
    name=unity_model_name,
    alias="champion",
    version=latest_version
)

print(f"Alias 'champion' set on version {latest_version} of {unity_model_name}")

## 9-3: Load and verify the registered model

In [0]:
# Load the champion model using its alias.
# This is the pattern downstream scoring jobs should use.
loaded_model = mlflow.spark.load_model(f"models:/{unity_model_name}@champion")

# Quick smoke-test: score a small sample and print a few predictions.
sample = test_data.limit(10)
sample_preds = loaded_model.transform(sample)
display(sample_preds.select("label", "prediction"))

# End of Notebook